In [6]:
import pandas as pd
import numpy as np

# Load datasets
nse = pd.read_csv('nsedata1.csv')
bse = pd.read_csv('bsedata1.csv')

def compute_historical_volatility(df, name="Dataset"):
    print(f"\n===== {name} =====\n")

    # Convert Date column to datetime if exists
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values(by='Date')

    # Drop non-numeric columns except Date
    numeric_df = df.select_dtypes(include=[np.number])

    # Take last ~1 month (21 trading days)
    last_month = numeric_df.tail(21)

    vol_dict = {}

    for col in last_month.columns:
        prices = last_month[col].dropna()

        # Compute log returns
        log_returns = np.log(prices / prices.shift(1)).dropna()

        # Daily volatility
        sigma_d = log_returns.std()

        # Annualized volatility
        sigma_a = sigma_d * np.sqrt(252)

        vol_dict[col] = sigma_a

        print(f"{col}: {sigma_a:.4f}")

    return vol_dict


# Compute for NSE
nse_vol = compute_historical_volatility(nse, "NSE Data")

# Compute for BSE
bse_vol = compute_historical_volatility(bse, "BSE Data")


===== NSE Data =====

^NSEI: 0.0736
RELIANCE.NS: 0.1056
TCS.NS: 0.1385
INFY.NS: 0.1767
HDFCBANK.NS: 0.1043
ICICIBANK.NS: 0.1054
LT.NS: 0.1207
ITC.NS: 0.0849
SBIN.NS: 0.1471
BHARTIARTL.NS: 0.1636
HINDUNILVR.NS: 0.2173
IDEA.NS: 0.4115
YESBANK.NS: 0.1760
SUZLON.NS: 0.2257
PNB.NS: 0.2844
IRFC.NS: 0.4758
FEDERALBNK.NS: 0.1406
BHEL.NS: 0.2586
SAIL.NS: 0.2933
IDFCFIRSTB.NS: 0.1987
UCOBANK.NS: 0.1800

===== BSE Data =====

^BSESN: 0.0695
RELIANCE.BO: 0.1051
TCS.BO: 0.1388
INFY.BO: 0.1742
HDFCBANK.BO: 0.1016
ICICIBANK.BO: 0.1045
LT.BO: 0.1193
ITC.BO: 0.0854
SBIN.BO: 0.1471
BHARTIARTL.BO: 0.1599
HINDUNILVR.BO: 0.2132
IDEA.BO: 0.4060
YESBANK.BO: 0.1795
SUZLON.BO: 0.2251
PNB.BO: 0.2889
IRFC.BO: 0.4703
FEDERALBNK.BO: 0.1432
BHEL.BO: 0.2610
SAIL.BO: 0.2924
IDFCFIRSTB.BO: 0.1987
UCOBANK.BO: 0.1814


In [3]:
import pandas as pd
import glob

# load all csv files
files = glob.glob("*.csv")

df_list = []

for file in files:
    df = pd.read_csv(file)

    # clean column names
    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        'Option type': 'Option Type',
        'Open Int': 'Open Interest'
    })
    # keep only useful columns
    df = df[['Date', 'Expiry', 'Strike Price', 'Option Type',
             'Close', 'Open Interest', 'Underlying Value']]

    df_list.append(df)

# combine all
final_df = pd.concat(df_list, ignore_index=True)

# sort data
final_df = final_df.sort_values(by=['Date', 'Expiry', 'Strike Price'])

# save to excel
final_df.to_excel("NIFTYoptiondata.xlsx", index=False)

print("Done!")

Done!


In [4]:
import pandas as pd

# load file
df = pd.read_excel("NIFTYoptiondata.xlsx")

# clean column names
df.columns = df.columns.str.strip()

# convert columns
df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%Y', errors='coerce')
df['Expiry'] = pd.to_datetime(df['Expiry'], format='%d-%b-%Y', errors='coerce')

df['Open Interest'] = pd.to_numeric(df['Open Interest'], errors='coerce')
df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
df['Underlying Value'] = pd.to_numeric(df['Underlying Value'], errors='coerce')

# sort
df = df.sort_values(by=['Date', 'Expiry', 'Strike Price'])

# save
df.to_excel("NIFTYoptiondata_cleaned.xlsx", index=False)

print("Final cleaned file ready!")

Final cleaned file ready!
